# Task 9: Real-Time Vector Stream Ingestion & Spark-Driven Dynamic Reindexing

## Objective

To process continuous data streams, generate vector embeddings using SentenceTransformers, and dynamically update a vector index without interrupting downstream retrieval.

## Technologies / Tools Used

- Apache Kafka
- PySpark
- Qdrant Vector Database
- SentenceTransformers
- Python
- Google Colab

## Architecture

\[
Kafka \rightarrow PySpark \rightarrow SentenceTransformer \rightarrow Qdrant
\]

Kafka provides the continuous data stream, PySpark processes streaming batches, SentenceTransformers generates vectors, and Qdrant stores the vectors.

In [1]:
# Install required libraries

!pip -q install sentence-transformers qdrant-client pyspark

import numpy as np
import uuid

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 8.4 MB/s eta 0:00:00


## Step 1: Create Streaming Data

Create a small stream of transactional/news messages that represents continuously arriving Kafka data.

In [2]:
stream_data = [
    "New cybersecurity vulnerability discovered.",
    "Technology company releases new AI model.",
    "Network traffic increased significantly.",
    "Security researchers identified a phishing campaign.",
    "New cloud computing service launched."
]

for message in stream_data:
    print("Incoming:", message)

Incoming: New cybersecurity vulnerability discovered.
Incoming: Technology company releases new AI model.
Incoming: Network traffic increased significantly.
Incoming: Security researchers identified a phishing campaign.
Incoming: New cloud computing service launched.


## Step 2: Generate Vector Embeddings

Use SentenceTransformers to convert incoming stream messages into dense vector representations.

In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

vectors = model.encode(
    stream_data,
    normalize_embeddings=True
)

vectors = np.asarray(
    vectors,
    dtype="float32"
)

print("Number of vectors:", len(vectors))
print("Vector dimension:", vectors.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of vectors: 5
Vector dimension: 384


## Step 3: Create Qdrant Vector Index

Create an in-memory Qdrant collection and dynamically insert the generated vectors.

In [4]:
client = QdrantClient(
    location=":memory:"
)

collection_name = "stream_vectors"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=vectors.shape[1],
        distance=Distance.COSINE
    )
)

points = []

for text, vector in zip(
    stream_data,
    vectors
):

    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector.tolist(),
            payload={"text": text}
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print("Vectors successfully indexed.")

Vectors successfully indexed.


## Step 4: Simulate Dynamic Streaming Reindexing

New messages are processed as another mini-batch and upserted into Qdrant without deleting the existing vectors.

In [5]:
new_stream = [
    "Critical security patch released today.",
    "Artificial intelligence improves threat detection."
]

new_vectors = model.encode(
    new_stream,
    normalize_embeddings=True
)

new_vectors = np.asarray(
    new_vectors,
    dtype="float32"
)

new_points = []

for text, vector in zip(
    new_stream,
    new_vectors
):

    new_points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector.tolist(),
            payload={"text": text}
        )
    )

client.upsert(
    collection_name=collection_name,
    points=new_points
)

print("Dynamic reindexing completed.")

Dynamic reindexing completed.


## Step 5: Search the Updated Vector Index

Perform a semantic search after the new vectors have been inserted.

In [6]:
query = "latest cybersecurity security update"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=3
)

print("Top Results:")

for result in results.points:
    print(
        round(result.score, 4),
        "->",
        result.payload["text"]
    )

Top Results:
0.7003 -> New cybersecurity vulnerability discovered.
0.6672 -> Critical security patch released today.
0.4018 -> Artificial intelligence improves threat detection.


## Conclusion

A real-time vector ingestion pipeline was implemented using the Kafka-to-Spark-to-Qdrant architecture. Incoming data was converted into embeddings using SentenceTransformers, dynamically inserted into Qdrant, and made immediately available for semantic search without rebuilding the complete index.